# 小市值效应因子分析 (Small Cap Factor Analysis)

使用 Alphalens 框架分析市值因子 `-log(tot_mv)`。此 Notebook 自动渲染所有图表及分析表格。

In [ ]:
import pandas as pd
import numpy as np
import alphalens as al
from alphalens_data import AlphalensData
import matplotlib.pyplot as plt

%matplotlib inline

# 1. 加载数据
data_path = "data/backend/l3_universe.parquet"
print(f"Loading data from {data_path}...")
df = pd.read_parquet(data_path)
df['date'] = pd.to_datetime(df['date'])

# 2. 筛选样本 (target_universe)
df_filtered = df[df['target_universe'] == True].copy()


# 3. 计算因子: 小市值因子 = -log(tot_mv)
df_filtered['factor'] = -np.log(df_filtered['tot_mv'])

# 4. 准备数据
factor = df_filtered.set_index(['date', 'symbol'])['factor']
prices = df_filtered.pivot(index='date', columns='symbol', values='close')
groupby = df_filtered.set_index(['date', 'symbol'])['industry_name'].astype(str)

print(f"Factor size: {len(factor)}")
print(f"Price shape: {prices.shape}")

In [ ]:
# 5. 使用 AlphalensData 包装器处理数据
analysis = AlphalensData(
    factor=factor,
    prices=prices,
    groupby=groupby,
    periods=(1, 5, 10),
    quantiles=5
)

factor_data = analysis.get_factor_data()

# 确保 group 列为字符串格式
if 'group' in factor_data.columns:
    factor_data['group'] = factor_data['group'].astype(str)

print("Cleaned factor data summary:")
print(analysis.summary())
factor_data.head()

In [ ]:
# 6. 输出完整分析报告 (交互式渲染)
# 包含：收益分析、IC分析、换手率分析
al.tears.create_full_tear_sheet(factor_data)